# 06 · Predictions from a neural network

Loads a saved bundle and scores unseen descriptions. Works for any of
the trained models — the bundle records which architecture it holds and
`score()` dispatches on that.

Three things this notebook will not do quietly:

- **It will not clean with anything but the saved cleaner.** A scoring
  path that cleans differently from training sees mostly
  out-of-vocabulary words and degrades silently rather than failing.
- **It will not score a vendor-marked model without the marker.** That
  would strip a token the model was trained to expect, from every row,
  with no error and a uniformly worse answer. Section 0 detects it.
- **It will not drop rows.** Every input comes back, including the ones
  that could not be read; those are flagged, not removed.

In [0]:
import pathlib, sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "nn_classifier").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

import nn_classifier as nnc
from nn_classifier import artifacts, paths, predict
print(paths.describe())

## 0 · Pick a bundle, and work out how it must be fed

`BUNDLE` is an explicit path rather than `paths.output_dir() / name`.
The arms that changed the cleaning wrote to their own directories so
they could not overwrite each other, which means the shared `outputs/`
does not contain them. Pointing directly avoids having to set
`NN_CLASSIFIER_OUTPUT_DIR` here — which would also redirect where this
notebook writes its predictions.

In [0]:
# ---------------------------------------------------------------------
# The three settings for this run.
# ---------------------------------------------------------------------

BUNDLE = ROOT / "outputs_gvendor" / "nn_gvendor_for_testpreds"
# Others, for reference:
#   ROOT / "outputs" / "nn_combined"        <- 10f, no vendor marker
#   ROOT / "outputs" / "nn_control_100"     <- 12c, no vendor marker
#   ROOT / "outputs_gvendor" / "nn_gvendor" <- 12b full run

# Column holding the supplier in the data being scored. Only used when
# the bundle was trained with a marker; ignored otherwise.
VENDOR_SOURCE_COL = "Global Vendno"

# The identifier in the file being scored. `score()` now raises if this
# is not a real column -- it used to be dropped silently, so the output
# came back with nothing to join on. Set to None to score without one.
ID_COL = "ID"

# Extra columns to carry through to the output, for joining or review.
CARRY_COLS = []

# True  -> every row keeps all four predicted levels.
# False -> the cascade blanks levels below the confidence each row
#          earned, and adds an `Assigned Depth` column.
ASSIGN_ALL = True

# ---------------------------------------------------------------------

if not BUNDLE.exists():
    raise FileNotFoundError(
        f"No bundle at {BUNDLE}.\n"
        f"  Directories present under {ROOT}:\n    "
        + "\n    ".join(sorted(
            str(p.relative_to(ROOT)) for p in ROOT.glob("outputs*") if p.is_dir()))
    )

print(artifacts.describe_bundle(BUNDLE))
bundle = artifacts.load_bundle(BUNDLE)
levels = list(bundle["hierarchy"].level_columns)

In [0]:
# Does this bundle need the vendor marker? Read it from the config the
# bundle was saved with, rather than remembering which arm produced it.
saved_cfg = bundle.get("config")
NEEDS_VENDOR = bool(getattr(getattr(saved_cfg, "cleaning", None),
                            "attach_vendor", False))
TRAINED_ON_COL = getattr(getattr(saved_cfg, "data", None), "vendor_col", None)

# `score(vendor_col=...)` takes the column name, or False to score
# without a marker at all.
VENDOR_COL = VENDOR_SOURCE_COL if NEEDS_VENDOR else False

print(f"trained with a vendor marker : {NEEDS_VENDOR}")
if NEEDS_VENDOR:
    print(f"  training used column       : {TRAINED_ON_COL!r}")
    print(f"  scoring will use           : {VENDOR_COL!r}")
    print("  attach_vendor_token strips and de-spaces the value itself,")
    print("  and the vectorizer lowercases, so the raw column is fine.")
else:
    print("  scoring without a marker")
print()
print(f"levels     : {levels}")
print(f"assign all : {ASSIGN_ALL}")

## 1 · A handful of descriptions, by hand

Raw text exactly as it arrives — no pre-cleaning. The bundle's cleaner
does that, which is the point.

The last two rows are deliberate: one with nothing readable, one with a
blank vendor. A blank vendor gets no marker at all rather than a bare
`VND_`, so it is the case worth seeing before trusting a batch.

In [0]:
samples = pd.DataFrame({
    "ID": [f"S{i}" for i in range(1, 9)],
    "Global Vendno": ["100123", "100123", "204881", "204881",
                      "311907", "311907", "100123", ""],
    "FullDesc": [
        "QJX8A TORQUE HEAD BOXEND 1-1/4X3-1/2",
        "VALVE BALL 2IN BRASS THREADED",
        "GLOVE NITRILE DISPOSABLE BLUE LARGE 100/BX",
        "SAFETY GLASSES CLEAR ANTI-FOG",
        "END MILL 1/2IN 4 FLUTE CARBIDE TIALN",
        "BEARING BALL DEEP GROOVE 6203-2RS",
        "XK9920-AA",                     # nothing readable
        "HOSE HYDRAULIC 1/2IN 3000PSI",  # no vendor
    ],
})

scored = predict.score(
    str(BUNDLE), samples,
    text_col="FullDesc", id_col="ID",   # the samples do have one
    vendor_col=VENDOR_COL,
    joint_mode="independent",
    predict_all_levels=True,
    use_cascade=not ASSIGN_ALL,
    min_known_tokens=2,
    bundle=bundle,
)

cols = (["ID", "FullDesc", "n_tokens", "known_share"]
        + [f"Predicted {c}" for c in levels]
        + [f"{c} Confidence" for c in levels]
        + ["Confidence", "Path Evidence", "Assigned Depth",
           "Needs Review", "Review Reason"])
scored[[c for c in cols if c in scored.columns]]

## 2 · A file

Chunked, because the memory cost here is the **probability matrices**,
not the model or the frame. One float per class per level per row —
4,384 floats, about 17.5 KB, on this taxonomy — and calibration makes a
second copy. At 600,000 rows that is roughly 21 GB in a single pass.

`score()` blocks by default and concatenates only the small output
frames. Chunking is exactly equivalent: nothing in the per-row path is
computed across rows.

In [0]:
TO_PREDICT = paths.data_path("to_predict.txt")

unlabelled = pd.read_csv(TO_PREDICT, delimiter="|", dtype="str")
print(f"{len(unlabelled):,} rows, columns: {list(unlabelled.columns)}")
print()

# Checked here rather than discovered by its absence in the output.
if ID_COL is not None and ID_COL not in unlabelled.columns:
    raise KeyError(
        f"ID_COL={ID_COL!r} is not in {TO_PREDICT}.\n"
        f"  Available: {list(unlabelled.columns)}\n"
        "  Set ID_COL to the right header, or None to score without one."
    )

# The check that decides whether this bundle can score this file.
if NEEDS_VENDOR:
    if VENDOR_SOURCE_COL not in unlabelled.columns:
        raise KeyError(
            f"{VENDOR_SOURCE_COL!r} is not in {TO_PREDICT}, but this bundle "
            "was trained with a vendor marker. Join the supplier in, or "
            "score this file with a no-marker bundle (10f / 12c) instead."
        )
    v = unlabelled[VENDOR_SOURCE_COL].fillna("").astype(str).str.strip()
    print(f"vendor present on : {(v != '').mean():.1%} of rows")
    print(f"distinct vendors  : {v[v != ''].nunique():,}")
    if (v != "").mean() < 0.9:
        print("  >> A blank vendor means no marker at all on that row, which")
        print("     is a shape the model saw rarely. Expect those rows to be")
        print("     weaker; `Needs Review` will not single them out.")

In [0]:
# What one block costs, before committing to the run.
block = predict.choose_chunk_size(bundle)
per_row = sum(bundle["hierarchy"].n_classes(i)
              for i in range(bundle["hierarchy"].n_levels))
print(f"block size  : {block:,} rows")
print(f"one pass    : {len(unlabelled) * per_row * 4 * 2 / 1e9:.1f} GB of "
      f"probabilities")
print(f"one block   : {block * per_row * 4 * 2 / 1e9:.2f} GB")
print()

scored = predict.score(
    str(BUNDLE), unlabelled,
    text_col="FullDesc", id_col=ID_COL,
    carry_cols=CARRY_COLS,
    vendor_col=VENDOR_COL,
    joint_mode="independent",
    predict_all_levels=True,
    use_cascade=not ASSIGN_ALL,
    min_known_tokens=2,
    bundle=bundle,          # loaded once; reloading per block dominates
)

assert len(scored) == len(unlabelled), "rows lost"
display(predict.score_summary(scored, levels))
scored.head()

### What came back

With `ASSIGN_ALL = True` every row carries a full four-level path, so
`Needs Review` is advisory rather than structural — nothing has been
blanked. Sort or filter on `Confidence` to decide what to act on.

In [0]:
import matplotlib.pyplot as plt

# The per-level level-4 confidence, NOT the joint `Confidence`.
# Temperature scaling was fitted per level, so the joint figure is a
# product of four calibrated numbers and compounds their error:
# measured on validation, joint ECE 0.052 against 0.025 for level 4.
CONF_COL = f"{levels[-1]} Confidence"
if CONF_COL not in scored.columns:
    CONF_COL = "Confidence"
    print("  >> per-level confidence absent; falling back to the joint one")
print(f"thresholding on: {CONF_COL}")

conf = scored[CONF_COL].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(conf, bins=40)
axes[0].set_xlabel("confidence"); axes[0].set_ylabel("rows")
axes[0].set_title("Confidence distribution")

# Volume above each threshold. Without labels there is no accuracy to
# plot here -- the accuracy that goes with each confidence band came
# from the validation run, in notebook 03's coverage table.
grid = np.round(np.arange(0.3, 1.0, 0.05), 2)
axes[1].plot(grid, [(conf >= t).mean() for t in grid], marker="o")
axes[1].set_xlabel("confidence threshold"); axes[1].set_ylabel("share kept")
axes[1].set_title("How much survives each cut")
axes[1].grid(alpha=0.3)
fig

In [0]:
band = pd.cut(conf, [0, .5, .6, .7, .8, .9, .95, 1.0], include_lowest=True)
summary = (pd.DataFrame({"band": band})
           .groupby("band", observed=False).size().to_frame("rows"))
summary["share"] = (summary["rows"] / len(scored)).round(4)
summary["cumulative from the top"] = (
    summary["rows"][::-1].cumsum()[::-1] / len(scored)).round(4)
display(summary)

print("Accuracy per band is NOT measurable here -- these rows have no")
print("labels. Read it off notebook 03's validation coverage table and")
print("apply it to these volumes.")

In [0]:
out = paths.output_path("predictions_nn.csv")
scored.to_csv(out, index=False)
print(f"wrote {len(scored):,} rows to {out}")
print(f"bundle: {BUNDLE.name}   vendor marker: {NEEDS_VENDOR}   "
      f"all levels: {ASSIGN_ALL}")

## If the output frame will not fit

`score()` still concatenates its blocks at the end. `score_to_csv()`
never does — it streams each block straight to disk, so peak memory is
one block whatever the input size.

```python
predict.score_to_csv(
    str(BUNDLE), unlabelled, str(paths.output_path("predictions_nn.csv")),
    text_col="FullDesc", id_col="ID",
    vendor_col=VENDOR_COL, use_cascade=not ASSIGN_ALL, bundle=bundle,
)
```

## What the output columns mean

| column | meaning |
|---|---|
| `Predicted Level N ID` / `Name` | the assigned path |
| `Confidence` | probability of the full path. **Worst-calibrated column here** — joint ECE 0.052 |
| `Level N ID Confidence` | that level's own calibrated probability for the class predicted. ECE 0.025 at level 4. **Threshold on this** |
| `Path Evidence` | the same score *un*-normalised |
| `Level N Prefix Probability` | mass summed over every path sharing that prefix |
| `Assigned Depth` | only present when `ASSIGN_ALL = False` |
| `Needs Review` / `Review Reason` | why a row would not have been auto-assigned |
| `n_tokens`, `known_share` | how much of the description the model could read |

**`Path Evidence` is the one that is easy to skip and worth keeping.**
A row where every path is implausible still produces a
confident-*looking* normalised score, because normalising divides by a
total that is itself tiny. The evidence figure is what gives that away,
and it is the column to sort by when spot-checking a batch.

## Settings that change what comes back

| | effect |
|---|---|
| `ASSIGN_ALL = False` | cascade blanks levels below the confidence each row earned |
| `vendor_col=False` | score a marked bundle without the marker — degrades every row; for diagnosis only |
| `min_known_tokens=0` | stop flagging unreadable rows. Cosmetic; nothing is blanked either way |
| `chunk_size=0` | single pass, the old behaviour. Only for small files |